In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)
    
@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)    

In [4]:

llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it",temperature=0)

agent = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [6]:

#lets see the memory of this agent
ask("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [7]:
#here we didnt give the name of the product..so the AI agent cant able to find it..
#In chatpgt ,it has some memory based on the pervious chats of us
#likewise we are also need to add some memory to our agent
ask("what are the reviews on this product")

[{'type': 'thinking', 'thinking': 'The user is asking for reviews on "this product", but they haven\'t specified which product they are referring to. I need to ask the user for the name of the product they are interested in.\n\nLooking at the available tools:\n- `get_product`: Gets price, rating, stock, and description.\n- `get_review`: Gets product name, number of reviews, and rating.\n\nBoth tools require a `name` parameter. Since the user didn\'t provide one, I cannot call either tool yet.\n\nTherefore, I should respond by asking the user to specify the product name.'}, {'type': 'text', 'text': "I'd be happy to help you with that! Could you please tell me the name of the product you're interested in?"}]


In [10]:
#for adding memory to the agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGoogleGenerativeAI(model="gemma-4-31b-it",temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-alice-session-1"}} #Which conversation's memory should I use? ..for understanding Each message have unique threadID,if User ask any ques from previous req..it decides which thread to use for current query fro user
    result = agent2.invoke({"messages": [{"role": "user", "content": question}], 
        },
        config=config   #This part actually sends the thread information to the agent
    )
    print(result["messages"][-1].content)



In [11]:
ask2("what is the price of wireless headphones.")

The price of the wireless headphones is $79.99.


In [12]:
ask2("What is the review of this product")

The wireless headphones have a rating of 4.6 based on 1,262 reviews.


In [15]:
ask2("what is the price of that product smart watch?")

The price of the smart watch is $199.99.


In [16]:

ask2("What is the price of the analog watch")

I'm sorry, but we don't have an analog watch in our store. We do have wireless headphones, a smart watch, a mechanical keyboard, and a laptop stand. Would you like information on any of those?


In [ ]:
agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="""
You are a helpful product assistant for an online tech store.

Rules:
1. Always use the get_product tool to find product information.
2. Never invent product names, prices, or availability.
3. If get_product says the product is not found, clearly say that the product is not available.
4. Do not suggest another product unless the user asks for alternatives.
5. If there is no valid price in the tool result, do not provide a price.
"""
)